# Phase 3 — AGCRN as a BasicTS 1.0 model
Run the cells **one by one** (Shift+Enter).
1. Check that our AGCRN port equals the official AGCRN code (forward output + gradients).
2. Run a short pipeline check (2 epochs) and export the result.

AGCRN is much heavier than STID. On **CPU** use the tiny preset (`TINY = True`, pipeline check only, a few minutes — my estimate, not measured).
On a **GPU** runtime set `GPUS = "0"` and `TINY = False` (the real model).

## 1. Which hardware do I have?

In [ ]:
!nvidia-smi 2>&1 | head -15
import sys
print(sys.version)

## 2. Mount Google Drive (a pop-up will ask for permission)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Get the repo and build the Python 3.11 environment (a few minutes; reused if it already exists)

In [ ]:
import os
%cd /content
if os.path.isdir("/content/proposal/.git"):
    !cd /content/proposal && git pull
else:
    !git clone --recursive https://github.com/hrfpour/proposal.git
%cd /content/proposal/code/BasicTS
if not os.path.exists("/content/venv/bin/python"):
    !pip -q install uv
    !uv venv /content/venv --python 3.11
    !uv pip install --python /content/venv/bin/python -r requirements.txt 2>&1 | tail -n 25
else:
    print("venv already exists - skipping install")

## 4. Make sure PyTorch exists in that environment, and that BasicTS imports

In [ ]:
import subprocess
PY = "/content/venv/bin/python"
if subprocess.run([PY, "-c", "import torch"], capture_output=True).returncode != 0:
    print("torch not in requirements.txt -> installing it")
    !uv pip install --python {PY} torch
!{PY} -c "import sys; sys.path.insert(0, '/content/proposal/code/BasicTS/src'); import basicts, torch, numpy; print('basicts', basicts.__version__, '| torch', torch.__version__, '| numpy', numpy.__version__, '| cuda', torch.cuda.is_available())"

## 5. Put the PEMS04 / PEMS08 data in place (from the PEMS-only copy cached in your Drive)

In [ ]:
import os, glob, shutil
KEEP = ["PEMS04", "PEMS08"]
CACHE = "/content/drive/MyDrive/proposal_data"
LOCAL_ZIP = "/content/datasets.zip"
DATA_URL = "PASTE_YOUR_LINK_HERE"   # only needed if the cache is missing
BT = "/content/proposal/code/BasicTS"
os.makedirs(f"{BT}/datasets", exist_ok=True)

if not all(os.path.isdir(f"{CACHE}/{n}") for n in KEEP):
    print("Cache missing -> downloading datasets.zip")
    assert DATA_URL.startswith("http"), "Edit DATA_URL first"
    !pip -q install gdown
    !gdown --fuzzy "{DATA_URL}" -O {LOCAL_ZIP}
    !rm -rf /content/_tmp && mkdir -p /content/_tmp
    !unzip -q -o {LOCAL_ZIP} '*PEMS04*' '*PEMS08*' -d /content/_tmp
    os.makedirs(CACHE, exist_ok=True)
    for n in KEEP:
        hits = [p for p in glob.glob(f"/content/_tmp/**/{n}", recursive=True) if os.path.isdir(p)]
        assert hits, f"Folder {n} not found inside the zip"
        shutil.copytree(hits[0], f"{CACHE}/{n}", dirs_exist_ok=True)

for n in KEEP:
    shutil.copytree(f"{CACHE}/{n}", f"{BT}/datasets/{n}", dirs_exist_ok=True)
!ls {BT}/datasets/PEMS04

## 6. Get the OFFICIAL AGCRN code (only used for the equivalence check)

In [ ]:
import os
if not os.path.isdir("/content/AGCRN_official"):
    !git clone --depth 1 https://github.com/LeiBAI/AGCRN /content/AGCRN_official
!ls /content/AGCRN_official/model

## 7. Equivalence check: our port vs the official code
Expected last line: `RESULT: PASS (port is equivalent to the official code)`. If it says FAIL, stop and send the output.

In [ ]:
PY = "/content/venv/bin/python"
!{PY} /content/proposal/code/scripts/check_agcrn_equivalence.py --official /content/AGCRN_official

## 8. Train AGCRN on PEMS04 (pipeline check)
`GPUS = "none"` = CPU, `"0"` = GPU. `TINY = True` = 8 units / 1 layer (CPU pipeline check); `False` = the real AGCRN (use a GPU).

In [ ]:
GPUS = "none"
TINY = True
EPOCHS = 2
PY = "/content/venv/bin/python"
TINY_FLAG = "--tiny" if TINY else ""
!{PY} /content/proposal/code/scripts/run_forecast.py --model agcrn --dataset PEMS04 --epochs {EPOCHS} --gpus {GPUS} {TINY_FLAG} 2>&1 | tee /content/run.log | tail -n 3

## 9. Show what the run printed and produced (send this output back)

In [ ]:
print("========== HEAD OF LOG ==========")
!head -n 12 /content/run.log
print("\n========== TAIL OF LOG ==========")
!tail -n 40 /content/run.log
print("\n========== FILES IN checkpoints ==========")
!find /content/proposal/code/BasicTS/checkpoints -type f | grep -v tensorboard | head -20

## 10. Export the result JSON and copy it to Drive

In [ ]:
EXPORT_STATUS = "smoke_test"
DEVICE = "cpu" if GPUS == "none" else "gpu"
!python /content/proposal/code/scripts/export_result.py --ckpt-root /content/proposal/code/BasicTS/checkpoints --model AGCRN --dataset PEMS04 --seed 42 --epochs {EPOCHS} --status {EXPORT_STATUS} --device {DEVICE} --out /content/results_out_agcrn
!ls /content/results_out_agcrn
import shutil, time
dest = f"/content/drive/MyDrive/proposal_runs/{time.strftime('%Y%m%d_%H%M')}_agcrn"
shutil.copytree("/content/results_out_agcrn", dest)
print("Saved to Drive:", dest)